# Invisible Distance
## Euclidean and Network Distance in Columbia University Food-Delivery Trips

**Author:** Yizhang Mu  
**Study area:** Restaurants within 5 km of Columbia University, New York City  
**Network:** Bicycle-accessible OpenStreetMap street network  
**Objects compared:** Restaurant origins and selected campus delivery destinations

---

## Research statement

Food-delivery platforms typically represent proximity through a simplified map interface, but a restaurant that appears close to a campus destination may require a substantially longer trip through the street network. This notebook defines a bicycle-accessible delivery network around Columbia University and measures distance between restaurant origins and campus destinations in two ways: **Euclidean distance**, the straight-line separation between two coordinates, and **network distance**, the length of the shortest feasible route through connected streets.

The project asks: **How does the structure of the street network reshape the experienced distance between restaurants and campus delivery destinations?** It tests whether straight-line proximity systematically underestimates routed travel, identifies where the greatest discrepancies occur, and interprets those discrepancies as an “invisible distance” produced by street geometry, barriers, access points, and network connectivity.


## Assignment response and research logic

This notebook directly addresses the assignment through four linked tasks:

1. **Define a network** — specify its nodes, edges, topology, and distance weight.
2. **Identify the measured elements** — restaurant origin nodes and campus destination nodes.
3. **Calculate distance** — compare Euclidean and shortest-path network distance for each origin–destination pair.
4. **Reflect on experience** — explain why the two distance measures imply different practical experiences of delivery accessibility.

The analysis follows a clear sequence:

> **Research statement → data preparation → network construction → node matching → distance calculation → comparison → reflection**


## Notebook roadmap

| Stage | Main question | Evidence produced |
|---|---|---|
| 1. Research frame | What is being measured, and why? | Research statement and definitions |
| 2. Real-world objects | Which restaurants and campus destinations are included? | Maps 1–4 and Charts 1–3 |
| 3. Network construction | How is the delivery network built? | Maps 5–8 and network statistics |
| 4. Distance calculation | How do Euclidean and network distance differ? | Maps 9–12 and Charts 5–6 |
| 5. Interpretation | Who experiences the greatest hidden distance, and why? | Maps 13–14, Charts 7–8, reflection |

Each map and chart serves a distinct analytical role; figures are not included merely to increase quantity.


## 0. Define the network

A network is represented as a weighted graph, **G = (V, E)**:

- **Nodes (V):** street intersections and endpoints in the bicycle-accessible street system.
- **Edges (E):** traversable street segments connecting those nodes.
- **Topology:** the pattern of connections that determines which movements are possible.
- **Edge weight:** physical segment length in meters.
- **Origin nodes:** the nearest network nodes to sampled restaurants.
- **Destination nodes:** the nearest network nodes to selected Columbia campus delivery locations.
- **Network distance:** the sum of edge lengths along the shortest feasible route between an origin node and a destination node.

The network therefore represents more than geographic location. It represents **permitted movement through connected urban infrastructure**.

### Distance measures

For each restaurant–destination pair:

- **Euclidean distance** measures direct straight-line separation.
- **Network distance** measures the shortest routed distance along the graph.
- **Hidden distance** = Network distance − Euclidean distance.
- **Detour ratio** = Network distance ÷ Euclidean distance.

A detour ratio of 1.00 means the route is almost as direct as a straight line. A larger value indicates that the network imposes additional travel.


## 1. Setup and visual system

In [ ]:
# Install packages once if your environment is missing them:
# %pip install -U osmnx geopandas pyogrio requests matplotlib

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import math
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import networkx as nx
import osmnx as ox

from shapely.geometry import Point, LineString
from IPython.display import display

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("networkx:", nx.__version__)
print("osmnx:", ox.__version__)

pd.set_option("display.max_columns", 30)
ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.requests_timeout = 180

mpl.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# A restrained visual system used across every figure.
COLORS = {
    "ink": "#17212B", "muted": "#8A96A3", "street": "#D9DEE3",
    "restaurant": "#D95D39", "campus": "#2C6E9B", "route": "#725AC1",
    "danger": "#B23A48", "highlight": "#F4B942", "background": "#F7F7F4"
}
plt.rcParams.update({
    "figure.facecolor": COLORS["background"], "axes.facecolor": COLORS["background"],
    "axes.titleweight": "bold", "axes.titlepad": 14, "axes.edgecolor": COLORS["muted"],
    "axes.labelcolor": COLORS["ink"], "xtick.color": COLORS["ink"], "ytick.color": COLORS["ink"],
    "font.family": "DejaVu Sans", "figure.dpi": 150, "savefig.dpi": 320
})

def finish_map(ax, title, subtitle=None, legend=True):
    ax.set_title(title, loc="left", fontsize=17, color=COLORS["ink"])
    if subtitle:
        ax.text(0, 1.01, subtitle, transform=ax.transAxes, fontsize=10, color=COLORS["muted"], va="bottom")
    ax.set_axis_off()
    if legend and ax.get_legend() is not None:
        ax.legend(frameon=False, loc="lower left")
    plt.tight_layout()

def finish_chart(ax, title, subtitle=None, xlabel=None, ylabel=None):
    ax.set_title(title, loc="left", fontsize=16, color=COLORS["ink"])
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=10, color=COLORS["muted"])
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=.18)
    plt.tight_layout()

## 2. Study design, measured nodes, and parameters

### Measured elements

The distance analysis connects two types of real-world objects:

- **Origins:** a reproducible, geographically stratified sample of restaurants located within 5 km of Columbia University.
- **Destinations:** selected campus buildings that function as plausible delivery endpoints.

The geographic coordinates of these objects are not automatically graph nodes. Each point is therefore **snapped to its nearest street-network node** before shortest-path calculation. The snap-distance diagnostic later checks whether this transformation is reasonable.


In [ ]:
STUDY_CENTER = (40.80754, -73.96257)  # Columbia University: (latitude, longitude)
STUDY_RADIUS_M = 5_000
NETWORK_RADIUS_M = 6_000
ROUTE_SAMPLE_SIZE = 80
RANDOM_SEED = 42
BICYCLE_SPEED_KMH = 15

# Campus destinations are explicitly defined and can be edited.
destination_records = [
    {"destination": "Avery Hall", "latitude": 40.80783, "longitude": -73.96266},
    {"destination": "Butler Library", "latitude": 40.80656, "longitude": -73.96312},
    {"destination": "Lerner Hall", "latitude": 40.80698, "longitude": -73.96442},
    {"destination": "Northwest Corner Building", "latitude": 40.80964, "longitude": -73.96148},
    {"destination": "Fayerweather Hall", "latitude": 40.80874, "longitude": -73.96095},
]

study_center = gpd.GeoDataFrame(
    {"name": ["Columbia University"]},
    geometry=[Point(STUDY_CENTER[1], STUDY_CENTER[0])],
    crs="EPSG:4326",
)
local_crs = study_center.estimate_utm_crs()
print("Local projected CRS:", local_crs)
print("Study radius:", STUDY_RADIUS_M / 1000, "km")

# Act I — The visible food landscape

Before calculating routes, the notebook establishes the scale and composition of the food environment. These figures answer **what exists** before asking **how it is connected**.

## 3. Build the real restaurant dataset

In [ ]:
api_url = "https://data.cityofnewyork.us/resource/43nn-pn8j.json"
cache_file = Path("nyc_restaurants_bbox_raw.csv")

# Bounding box is intentionally larger than the final 5 km circular filter.
lat_pad = 0.060
lon_pad = 0.080
lat_min, lat_max = STUDY_CENTER[0] - lat_pad, STUDY_CENTER[0] + lat_pad
lon_min, lon_max = STUDY_CENTER[1] - lon_pad, STUDY_CENTER[1] + lon_pad

select_cols = (
    "camis,dba,boro,building,street,zipcode,cuisine_description,"
    "latitude,longitude,grade,inspection_date"
)

# Paginate instead of assuming that one request returns every record.
# latitude and longitude are stored as text in this Socrata dataset, so the
# broad API request is followed by a precise projected 5 km filter below.
def download_restaurant_records(page_size=50000, max_pages=8):
    pages = []
    for page in range(max_pages):
        params = {
            "$select": select_cols,
            "$where": "latitude IS NOT NULL AND longitude IS NOT NULL",
            "$limit": page_size,
            "$offset": page * page_size,
            "$order": "camis,inspection_date DESC",
        }
        response = requests.get(api_url, params=params, timeout=180)
        response.raise_for_status()
        batch = response.json()
        if not batch:
            break
        pages.append(pd.DataFrame(batch))
        print(f"Downloaded page {page + 1}: {len(batch):,} records")
        if len(batch) < page_size:
            break
    if not pages:
        raise RuntimeError("The NYC Open Data API returned no restaurant records.")
    return pd.concat(pages, ignore_index=True)

try:
    raw_restaurants = download_restaurant_records()
    raw_restaurants.to_csv(cache_file, index=False)
    print(f"Saved {len(raw_restaurants):,} inspection records to {cache_file}.")
except Exception as exc:
    if cache_file.exists():
        raw_restaurants = pd.read_csv(cache_file, low_memory=False)
        print(f"API unavailable; loaded {len(raw_restaurants):,} cached records.")
    else:
        raise RuntimeError(
            "Restaurant download failed and no local cache exists. Check the internet "
            "connection, run the package-install cell, and rerun this cell."
        ) from exc

In [ ]:
restaurants = raw_restaurants.copy()
for col in ["latitude", "longitude"]:
    restaurants[col] = pd.to_numeric(restaurants[col], errors="coerce")
restaurants["inspection_date"] = pd.to_datetime(restaurants["inspection_date"], errors="coerce")
restaurants = restaurants.dropna(subset=["camis", "dba", "latitude", "longitude"])

# Apply a fast bounding-box filter before the exact metric-distance filter.
restaurants = restaurants[
    restaurants["latitude"].between(lat_min, lat_max)
    & restaurants["longitude"].between(lon_min, lon_max)
].copy()

# Keep the newest available inspection-linked record for each restaurant permit.
restaurants = (
    restaurants.sort_values("inspection_date", ascending=False)
    .drop_duplicates(subset="camis", keep="first")
)

restaurants_gdf = gpd.GeoDataFrame(
    restaurants,
    geometry=gpd.points_from_xy(restaurants.longitude, restaurants.latitude),
    crs="EPSG:4326",
)
restaurants_metric = restaurants_gdf.to_crs(local_crs)
center_metric = study_center.to_crs(local_crs).geometry.iloc[0]
restaurants_metric["distance_to_columbia_m"] = restaurants_metric.geometry.distance(center_metric)
restaurants_metric = restaurants_metric[
    restaurants_metric["distance_to_columbia_m"] <= STUDY_RADIUS_M
].copy()

if restaurants_metric.empty:
    raise RuntimeError("No restaurants remained after the 5 km filter. Inspect the downloaded coordinates.")

restaurants_gdf = restaurants_metric.to_crs("EPSG:4326")

bins = [0, 1000, 2000, 3000, 4000, 5000]
labels = ["0–1 km", "1–2 km", "2–3 km", "3–4 km", "4–5 km"]
restaurants_metric["distance_band"] = pd.cut(
    restaurants_metric["distance_to_columbia_m"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True,
)
restaurants_gdf["distance_to_columbia_m"] = restaurants_metric["distance_to_columbia_m"].to_numpy()
restaurants_gdf["distance_band"] = restaurants_metric["distance_band"].astype("string").to_numpy()

print(f"Unique real restaurants within 5 km: {len(restaurants_gdf):,}")
print(f"Unique cuisines: {restaurants_gdf['cuisine_description'].nunique(dropna=True):,}")
display(restaurants_gdf[["dba", "cuisine_description", "grade", "distance_to_columbia_m"]].head())

### Data-quality audit

In [ ]:
quality = pd.Series({
    "Unique restaurants": len(restaurants_gdf),
    "Cuisine categories": restaurants_gdf["cuisine_description"].nunique(dropna=True),
    "Missing cuisine labels": restaurants_gdf["cuisine_description"].isna().sum(),
    "Missing grades": restaurants_gdf["grade"].isna().sum(),
    "Duplicate permits": restaurants_gdf["camis"].duplicated().sum(),
})
display(quality.to_frame("value"))

## Map 1 — Study frame: a true 5 km radius

In [ ]:
ring = gpd.GeoSeries([center_metric.buffer(STUDY_RADIUS_M)], crs=local_crs)
fig, ax = plt.subplots(figsize=(10,10))
ring.plot(ax=ax, color="#E8ECEF", edgecolor=COLORS["ink"], linewidth=1.4, alpha=.65)
restaurants_metric.plot(ax=ax, color=COLORS["restaurant"], markersize=5, alpha=.35, label=f"{len(restaurants_metric):,} restaurants")
study_center.to_crs(local_crs).plot(ax=ax, color=COLORS["campus"], marker="*", markersize=220, edgecolor="white", linewidth=1.2, label="Columbia University")
finish_map(ax, "Map 1. The 5 km food-delivery study area", "All unique DOHMH restaurant permits retained after projected-distance filtering")
plt.show()

## Map 2 — Restaurant density as an urban field

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
xy=np.c_[restaurants_metric.geometry.x, restaurants_metric.geometry.y]
hb=ax.hexbin(xy[:,0],xy[:,1],gridsize=48,mincnt=1,cmap="magma",linewidths=0,alpha=.9)
ring.boundary.plot(ax=ax,color=COLORS["muted"],linewidth=.8)
study_center.to_crs(local_crs).plot(ax=ax,color="white",marker="*",markersize=180,edgecolor=COLORS["ink"],linewidth=1)
cb=fig.colorbar(hb,ax=ax,shrink=.72,pad=.02); cb.set_label("Restaurants per hexagon")
finish_map(ax,"Map 2. Restaurant density is highly uneven","Hotspots show where platform-visible food supply clusters spatially",legend=False)
plt.show()

## Chart 1 — Cuisine composition

In [ ]:
counts=restaurants_gdf["cuisine_description"].fillna("Unknown").value_counts().head(15).sort_values()
fig,ax=plt.subplots(figsize=(10,7)); ax.barh(counts.index,counts.values,color=COLORS["restaurant"])
for i,v in enumerate(counts.values): ax.text(v,i,f"  {v:,}",va="center",fontsize=9)
finish_chart(ax,"Chart 1. The 15 most common cuisine categories","Categories are based on the latest retained DOHMH record",xlabel="Unique restaurants")
plt.show()

## Map 3 — Geography of the six dominant cuisines

In [ ]:
top6=restaurants_gdf["cuisine_description"].value_counts().head(6).index
m=restaurants_metric.copy(); m["group"]=np.where(m["cuisine_description"].isin(top6),m["cuisine_description"],"Other")
fig,ax=plt.subplots(figsize=(10,10)); ring.plot(ax=ax,color="#ECEFED",edgecolor="none")
m[m.group=="Other"].plot(ax=ax,color=COLORS["street"],markersize=4,alpha=.22)
for name in top6:
    m[m.group==name].plot(ax=ax,markersize=12,alpha=.7,label=name)
study_center.to_crs(local_crs).plot(ax=ax,color=COLORS["campus"],marker="*",markersize=170,edgecolor="white")
finish_map(ax,"Map 3. Cuisine categories occupy distinct spatial territories","Only the six largest groups are highlighted; all others remain as context")
plt.show()

## Chart 2 — Restaurant supply by distance band

In [ ]:
labels=["0–1 km","1–2 km","2–3 km","3–4 km","4–5 km"]
band=restaurants_metric["distance_band"].value_counts().reindex(labels).fillna(0)
fig,ax=plt.subplots(figsize=(9,5)); ax.bar(band.index,band.values,color=COLORS["campus"])
for i,v in enumerate(band.values): ax.text(i,v,f"{int(v):,}",ha="center",va="bottom",fontsize=9)
finish_chart(ax,"Chart 2. Restaurant supply expands with distance","Counts reflect the increasing area of successive rings",xlabel="Distance from Columbia",ylabel="Restaurants")
plt.show()

## 4. Select reproducible restaurant-origin nodes

In [ ]:
def proportional_stratified_sample(gdf, strata_col, n, seed=42):
    valid = gdf.dropna(subset=[strata_col]).copy()
    shares = valid[strata_col].value_counts(normalize=True)
    allocations = (shares * n).round().astype(int).clip(lower=1)
    # Correct rounding to exactly n.
    while allocations.sum() > n:
        idx = allocations.idxmax(); allocations.loc[idx] -= 1
    while allocations.sum() < n:
        idx = shares.idxmax(); allocations.loc[idx] += 1
    parts=[]
    for group, count in allocations.items():
        pool = valid[valid[strata_col] == group]
        parts.append(pool.sample(n=min(count, len(pool)), random_state=seed))
    sampled = pd.concat(parts).drop_duplicates("camis")
    if len(sampled) < n:
        remaining = valid[~valid.camis.isin(sampled.camis)]
        sampled = pd.concat([sampled, remaining.sample(n=min(n-len(sampled),len(remaining)), random_state=seed)])
    return sampled.head(n).copy()

route_restaurants = proportional_stratified_sample(
    restaurants_metric, "distance_band", ROUTE_SAMPLE_SIZE, RANDOM_SEED
)
route_restaurants_wgs = route_restaurants.to_crs("EPSG:4326")
print(f"Route-analysis restaurants: {len(route_restaurants):,}")
display(route_restaurants["distance_band"].value_counts().reindex(labels).to_frame("Sample count"))

## Map 4 — The sample preserves the full radial structure

In [ ]:
fig,ax=plt.subplots(figsize=(10,10)); ring.plot(ax=ax,color="#EEF0F2",edgecolor=COLORS["muted"],linewidth=.8)
restaurants_metric.plot(ax=ax,color=COLORS["street"],markersize=3,alpha=.25,label="All restaurants")
route_restaurants.plot(ax=ax,column="distance_band",categorical=True,markersize=28,alpha=.9,legend=True,edgecolor="white",linewidth=.25)
study_center.to_crs(local_crs).plot(ax=ax,color=COLORS["campus"],marker="*",markersize=180,edgecolor="white")
finish_map(ax,"Map 4. Stratified route-analysis sample","The sample is distributed across all five 1 km bands rather than concentrated near campus")
plt.show()

## Chart 3 — Full dataset versus route sample

In [ ]:
full=restaurants_metric["distance_band"].value_counts(normalize=True).reindex(labels).fillna(0)*100
sample=route_restaurants["distance_band"].value_counts(normalize=True).reindex(labels).fillna(0)*100
x=np.arange(len(labels)); w=.36
fig,ax=plt.subplots(figsize=(9,5)); ax.bar(x-w/2,full,w,label="Full dataset",color=COLORS["muted"]); ax.bar(x+w/2,sample,w,label="Route sample",color=COLORS["restaurant"])
ax.set_xticks(x,labels); ax.legend(frameon=False)
finish_chart(ax,"Chart 3. Sampling fidelity by distance band","Similar shares indicate that the routing sample preserves the radial distribution",ylabel="Share of restaurants (%)")
plt.show()

# Act II — The hidden network

The second act replaces apparent proximity with networked movement. It moves from street structure, to snapping, to route geometry, and finally to the corridors shared by many deliveries.

## 5. Construct the bicycle-accessible street network

In [ ]:
graph_file = Path("columbia_6km_bike_network.graphml")

try:
    G = ox.graph.graph_from_point(
        STUDY_CENTER,
        dist=NETWORK_RADIUS_M,
        network_type="bike",
        simplify=True,
        retain_all=False,
    )
    # Newer OSMnx versions already include edge lengths. Add them only if absent.
    if not all("length" in data for _, _, _, data in G.edges(keys=True, data=True)):
        G = ox.distance.add_edge_lengths(G)
    ox.io.save_graphml(G, graph_file)
    print("Downloaded and cached the bicycle network.")
except Exception as exc:
    if graph_file.exists():
        G = ox.io.load_graphml(graph_file)
        print("OSM download unavailable; loaded the cached bicycle network.")
    else:
        raise RuntimeError(
            "OpenStreetMap download failed and no cached GraphML file exists. "
            "Check the internet connection and confirm that OSMnx is installed."
        ) from exc

nodes, edges = ox.convert.graph_to_gdfs(G)
G_proj = ox.projection.project_graph(G, to_crs=local_crs)
nodes_proj, edges_proj = ox.convert.graph_to_gdfs(G_proj)
print(f"Network nodes: {len(nodes):,}; edges: {len(edges):,}")

## Map 5 — Network structure and study boundary

In [ ]:
fig,ax=plt.subplots(figsize=(11,11)); edges_proj.plot(ax=ax,color=COLORS["street"],linewidth=.42,alpha=.9)
ring.boundary.plot(ax=ax,color=COLORS["ink"],linewidth=1.2,linestyle="--")
study_center.to_crs(local_crs).plot(ax=ax,color=COLORS["campus"],marker="*",markersize=190,edgecolor="white",linewidth=1)
finish_map(ax,"Map 5. Bicycle-accessible street network","The network extends beyond the 5 km restaurant boundary so edge routes are not artificially clipped",legend=False)
plt.show()

## Map 6 — Network hierarchy by street length

In [ ]:
e=edges_proj.copy(); e["len_class"]=pd.qcut(e["length"],4,labels=["Short","Medium","Long","Very long"],duplicates="drop")
fig,ax=plt.subplots(figsize=(11,11));
for label_,lw,a in [("Short",.25,.25),("Medium",.45,.4),("Long",.8,.6),("Very long",1.3,.85)]:
    if label_ in e["len_class"].astype(str).values: e[e["len_class"].astype(str)==label_].plot(ax=ax,color=COLORS["ink"],linewidth=lw,alpha=a,label=label_)
ring.boundary.plot(ax=ax,color=COLORS["restaurant"],linewidth=1)
finish_map(ax,"Map 6. Edge length reveals the network’s grain","Longer segments form continuous avenues; shorter segments register finer local permeability")
plt.show()

## 6. Convert geographic objects into network nodes

In [ ]:
destinations = gpd.GeoDataFrame(
    destination_records,
    geometry=gpd.points_from_xy(
        [d["longitude"] for d in destination_records],
        [d["latitude"] for d in destination_records],
    ),
    crs="EPSG:4326",
)
destinations_metric = destinations.to_crs(local_crs)

route_restaurants_wgs = route_restaurants.to_crs("EPSG:4326").copy()
route_restaurants_wgs["network_node"] = ox.distance.nearest_nodes(
    G,
    X=route_restaurants_wgs.geometry.x.to_numpy(),
    Y=route_restaurants_wgs.geometry.y.to_numpy(),
)
destinations["network_node"] = ox.distance.nearest_nodes(
    G,
    X=destinations.geometry.x.to_numpy(),
    Y=destinations.geometry.y.to_numpy(),
)

route_restaurants = route_restaurants.copy()
route_restaurants["network_node"] = route_restaurants_wgs["network_node"].to_numpy()
destinations_metric["network_node"] = destinations["network_node"].to_numpy()

node_points = nodes_proj.geometry
route_restaurants["snap_distance_m"] = [
    geom.distance(node_points.loc[node])
    for geom, node in zip(route_restaurants.geometry, route_restaurants.network_node)
]
destinations_metric["snap_distance_m"] = [
    geom.distance(node_points.loc[node])
    for geom, node in zip(destinations_metric.geometry, destinations_metric.network_node)
]

print("Mean restaurant snap distance:", round(route_restaurants.snap_distance_m.mean(), 1), "m")
print("Maximum restaurant snap distance:", round(route_restaurants.snap_distance_m.max(), 1), "m")

## Map 7 — Snap-distance quality control

In [ ]:
snap_lines=[LineString([r.geometry,node_points.loc[r.network_node]]) for _,r in route_restaurants.iterrows()]
snap=gpd.GeoDataFrame({"snap_m":route_restaurants.snap_distance_m.values},geometry=snap_lines,crs=local_crs)
fig,ax=plt.subplots(figsize=(10,10)); edges_proj.plot(ax=ax,color=COLORS["street"],linewidth=.35)
snap.plot(ax=ax,column="snap_m",cmap="plasma",linewidth=1.4,legend=True)
route_restaurants.plot(ax=ax,color=COLORS["restaurant"],markersize=20,edgecolor="white",linewidth=.3)
finish_map(ax,"Map 7. Each restaurant is attached to its nearest network node","Long snap lines flag potential positional or network-access mismatches",legend=False)
plt.show()

## Chart 4 — Snap-distance distribution

In [ ]:
fig,ax=plt.subplots(figsize=(9,5)); ax.hist(route_restaurants.snap_distance_m,bins=20,color=COLORS["campus"],edgecolor="white")
ax.axvline(route_restaurants.snap_distance_m.median(),color=COLORS["danger"],linestyle="--",label=f"Median {route_restaurants.snap_distance_m.median():.1f} m")
ax.legend(frameon=False); finish_chart(ax,"Chart 4. Most points attach closely to the street network","This quality check prevents large snapping errors from being mistaken for delivery friction",xlabel="Snap distance (m)",ylabel="Restaurants")
plt.show()

## Map 8 — Campus destinations and their access nodes

In [ ]:
fig,ax=plt.subplots(figsize=(8,8)); edges_proj.cx[destinations_metric.total_bounds[0]-700:destinations_metric.total_bounds[2]+700,destinations_metric.total_bounds[1]-700:destinations_metric.total_bounds[3]+700].plot(ax=ax,color=COLORS["street"],linewidth=.55)
destinations_metric.plot(ax=ax,color=COLORS["campus"],markersize=90,edgecolor="white",linewidth=1,label="Campus destinations")
for _,r in destinations_metric.iterrows(): ax.annotate(r.destination,(r.geometry.x,r.geometry.y),xytext=(5,5),textcoords="offset points",fontsize=8)
finish_map(ax,"Map 8. Five campus destinations anchor the route comparison","Each destination represents a different position within the campus edge and interior")
plt.show()

## 7. Calculate Euclidean and network distance

### Step-by-step distance workflow

For every sampled restaurant and campus destination:

1. Read the restaurant and destination coordinates.
2. Project the coordinates into a local metric coordinate reference system.
3. Calculate Euclidean distance between the original point geometries.
4. Match each point to the nearest graph node.
5. Use Dijkstra’s shortest-path algorithm with edge length as the weight.
6. Sum the lengths of all edges along the resulting route.
7. Calculate hidden distance and detour ratio.
8. Store both the numeric results and route geometry for mapping.

This workflow keeps the conceptual distinction clear: Euclidean distance is calculated in continuous space, whereas network distance is calculated through a constrained graph.


In [ ]:
pair_rows = []
route_geometries = []
euclidean_geometries = []

for _, r in route_restaurants.iterrows():
    for _, d in destinations_metric.iterrows():
        euclidean_m = r.geometry.distance(d.geometry)
        try:
            route_nodes = nx.shortest_path(
                G_proj, r.network_node, d.network_node, weight="length"
            )
            network_m = nx.shortest_path_length(
                G_proj, r.network_node, d.network_node, weight="length"
            )
            route_edges = ox.routing.route_to_gdf(G_proj, route_nodes, weight="length")
            merged_coords = []
            for geom in route_edges.geometry:
                if geom.geom_type == "MultiLineString":
                    parts = list(geom.geoms)
                else:
                    parts = [geom]
                for part in parts:
                    coords = list(part.coords)
                    if merged_coords and coords and merged_coords[-1] == coords[0]:
                        coords = coords[1:]
                    merged_coords.extend(coords)
            route_line = LineString(merged_coords) if len(merged_coords) >= 2 else None
        except (nx.NetworkXNoPath, nx.NodeNotFound, ValueError, KeyError):
            network_m = np.nan
            route_line = None

        pair_rows.append({
            "camis": r.camis,
            "restaurant": r.dba,
            "cuisine": r.cuisine_description,
            "distance_band": str(r.distance_band),
            "destination": d.destination,
            "euclidean_m": euclidean_m,
            "network_m": network_m,
            "origin_node": r.network_node,
            "destination_node": d.network_node,
        })
        route_geometries.append(route_line)
        euclidean_geometries.append(LineString([r.geometry, d.geometry]))

pairs = gpd.GeoDataFrame(pair_rows, geometry=route_geometries, crs=local_crs)
pairs["euclidean_geometry"] = euclidean_geometries
pairs = pairs.dropna(subset=["network_m", "geometry"]).copy()

if pairs.empty:
    raise RuntimeError("No valid network routes were found. Try network_type='walk' or increase NETWORK_RADIUS_M.")

pairs["friction_m"] = pairs.network_m - pairs.euclidean_m
pairs["detour_ratio"] = pairs.network_m / pairs.euclidean_m.replace(0, np.nan)
pairs["modeled_minutes"] = pairs.network_m / (BICYCLE_SPEED_KMH * 1000 / 60)
pairs["extra_minutes"] = pairs.friction_m / (BICYCLE_SPEED_KMH * 1000 / 60)

# CSV cannot store Shapely objects directly, so export only analytical fields.
csv_export = pd.DataFrame(pairs.drop(columns=["geometry", "euclidean_geometry"]))
csv_export.to_csv("delivery_pairs_5km.csv", index=False)

# GeoJSON supports one active geometry column. Drop the auxiliary Euclidean geometry.
route_export = pairs.drop(columns=["euclidean_geometry"]).to_crs("EPSG:4326")
route_export.to_file("delivery_routes_5km.geojson", driver="GeoJSON", index=False)

print(f"Successful route pairs: {len(pairs):,}")
display(pairs[["restaurant", "destination", "euclidean_m", "network_m", "friction_m", "detour_ratio"]].head())

## Map 9 — Euclidean baseline: what the interface implies

In [ ]:
eucl=gpd.GeoDataFrame(pairs.drop(columns=["geometry","euclidean_geometry"]),geometry=list(pairs.euclidean_geometry),crs=local_crs)
fig,ax=plt.subplots(figsize=(11,11)); edges_proj.plot(ax=ax,color=COLORS["street"],linewidth=.18,alpha=.35)
eucl.plot(ax=ax,color=COLORS["highlight"],linewidth=.45,alpha=.18)
route_restaurants.plot(ax=ax,color=COLORS["restaurant"],markersize=10,alpha=.8); destinations_metric.plot(ax=ax,color=COLORS["campus"],markersize=80,edgecolor="white")
finish_map(ax,"Map 9. Straight-line proximity compresses the city","Every restaurant–destination pair is shown as a direct connection",legend=False)
plt.show()

## Map 10 — Routed reality: shortest bicycle paths

In [ ]:
fig,ax=plt.subplots(figsize=(11,11)); edges_proj.plot(ax=ax,color=COLORS["street"],linewidth=.16,alpha=.28)
pairs.plot(ax=ax,color=COLORS["route"],linewidth=.55,alpha=.22)
route_restaurants.plot(ax=ax,color=COLORS["restaurant"],markersize=10); destinations_metric.plot(ax=ax,color=COLORS["campus"],markersize=80,edgecolor="white")
finish_map(ax,"Map 10. The street network bends, bundles, and redirects movement","Shortest paths replace direct lines with operational delivery routes",legend=False)
plt.show()

## 8. Examine shared network structure

In [ ]:
# Count how often sampled routes use each directed network edge.
edge_use = {}
for _, row in pairs.iterrows():
    try:
        path = nx.shortest_path(
            G_proj, row.origin_node, row.destination_node, weight="length"
        )
        for u, v in zip(path[:-1], path[1:]):
            edge_data = G_proj.get_edge_data(u, v)
            if not edge_data:
                continue
            key = min(edge_data, key=lambda k: float(edge_data[k].get("length", np.inf)))
            idx = (u, v, key)
            edge_use[idx] = edge_use.get(idx, 0) + 1
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        continue

corridors = edges_proj.copy()
corridors["route_count"] = [edge_use.get(idx, 0) for idx in corridors.index]
corridors = corridors[corridors.route_count > 0].copy()

fig, ax = plt.subplots(figsize=(11, 11))
edges_proj.plot(ax=ax, color=COLORS["street"], linewidth=.18, alpha=.30)
if not corridors.empty:
    widths = 0.35 + corridors.route_count / corridors.route_count.max() * 5
    corridors.plot(ax=ax, column="route_count", linewidth=widths, legend=True)
destinations_metric.plot(ax=ax, color=COLORS["campus"], markersize=80, edgecolor="white")
finish_map(ax,"Map 11. Shared delivery corridors","Line width and tone identify streets repeatedly used across sampled routes",legend=False)
plt.show()

## Chart 5 — Euclidean and network distance

In [ ]:
fig,ax=plt.subplots(figsize=(8,7)); ax.scatter(pairs.euclidean_m,pairs.network_m,s=24,alpha=.38,color=COLORS["route"],edgecolor="none")
lim=max(pairs.euclidean_m.max(),pairs.network_m.max()); ax.plot([0,lim],[0,lim],linestyle="--",color=COLORS["danger"],label="Equal distance")
ax.legend(frameon=False); finish_chart(ax,"Chart 5. Routed distance consistently exceeds straight-line distance","Vertical distance above the diagonal is the hidden distance created by the network",xlabel="Euclidean distance (m)",ylabel="Network distance (m)")
plt.show()

# Act III — Unequal accessibility

The final act shifts from route geometry to distributional effects: how large is the detour, where is it concentrated, which destinations are burdened, and which restaurants change rank after the network is considered?

## Map 12 — Route-level detour ratio

In [ ]:
fig,ax=plt.subplots(figsize=(11,11)); edges_proj.plot(ax=ax,color=COLORS["street"],linewidth=.16,alpha=.25)
pairs.plot(ax=ax,column="detour_ratio",cmap="inferno",linewidth=1.0,alpha=.58,legend=True,legend_kwds={"label":"Network ÷ Euclidean"})
destinations_metric.plot(ax=ax,color=COLORS["campus"],markersize=80,edgecolor="white")
finish_map(ax,"Map 12. Detour is spatially selective","Bright routes require the largest proportional deviation from direct distance",legend=False)
plt.show()

## Chart 6 — Detour-ratio distribution

In [ ]:
cap=pairs.detour_ratio.quantile(.99); fig,ax=plt.subplots(figsize=(9,5)); ax.hist(pairs.detour_ratio.clip(upper=cap),bins=30,color=COLORS["danger"],edgecolor="white")
ax.axvline(pairs.detour_ratio.median(),color=COLORS["ink"],linestyle="--",label=f"Median {pairs.detour_ratio.median():.2f}×"); ax.legend(frameon=False)
finish_chart(ax,"Chart 6. Hidden distance is not a single constant","The distribution separates ordinary detours from exceptional network penalties",xlabel="Detour ratio",ylabel="Route pairs")
plt.show()

## 9. Compare destination-level accessibility

In [ ]:
destination_summary=(pairs.groupby("destination")
    .agg(route_pairs=("restaurant","count"),mean_euclidean_m=("euclidean_m","mean"),
         mean_network_m=("network_m","mean"),mean_friction_m=("friction_m","mean"),
         median_detour_ratio=("detour_ratio","median"),mean_extra_minutes=("extra_minutes","mean"))
    .reset_index())
destination_summary.to_csv("destination_accessibility_5km.csv",index=False)
display(destination_summary.sort_values("mean_friction_m",ascending=False))

## Map 13 — Campus-scale destination burden

This figure deliberately shifts from the regional 5 km study area to the **campus scale**. The map answers *where* the destinations are, while the compact ranking answers *how much* their mean hidden distance differs.

In [ ]:
dest_access = destinations_metric.merge(destination_summary, on="destination", how="left").copy()

# Short labels keep the campus-scale figure readable.
short_names = {
    "Avery Hall": "Avery",
    "Butler Library": "Butler",
    "Lerner Hall": "Lerner",
    "Northwest Corner Building": "NWC",
    "Fayerweather Hall": "Fayerweather",
}
dest_access["short_name"] = dest_access["destination"].map(short_names).fillna(dest_access["destination"])
dest_access = dest_access.sort_values("mean_friction_m", ascending=False).reset_index(drop=True)
dest_access["map_id"] = np.arange(1, len(dest_access) + 1)

# Zoom to the campus rather than showing the entire 5 km network.
minx, miny, maxx, maxy = dest_access.total_bounds
pad_x = max(280, (maxx - minx) * 2.8)
pad_y = max(280, (maxy - miny) * 2.8)
x0, x1 = minx - pad_x, maxx + pad_x
y0, y1 = miny - pad_y, maxy + pad_y
local_edges = edges_proj.cx[x0:x1, y0:y1]

# Marker area represents mean hidden distance without an oversized colorbar.
v = dest_access["mean_friction_m"].astype(float)
if np.isclose(v.max(), v.min()):
    marker_sizes = np.full(len(v), 360.0)
else:
    marker_sizes = 250 + 420 * (v - v.min()) / (v.max() - v.min())

fig = plt.figure(figsize=(12.5, 7.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1.7, 1.0], wspace=0.10)
ax_map = fig.add_subplot(gs[0, 0])
ax_rank = fig.add_subplot(gs[0, 1])

# Campus detail map.
local_edges.plot(ax=ax_map, color="#D7DDE2", linewidth=0.70, alpha=0.78, zorder=1)
ax_map.scatter(
    dest_access.geometry.x,
    dest_access.geometry.y,
    s=marker_sizes,
    color=COLORS["campus"],
    edgecolor="white",
    linewidth=2.0,
    alpha=0.95,
    zorder=3,
)

for _, row in dest_access.iterrows():
    ax_map.text(
        row.geometry.x,
        row.geometry.y,
        str(int(row.map_id)),
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="white",
        zorder=4,
    )

ax_map.set_xlim(x0, x1)
ax_map.set_ylim(y0, y1)
ax_map.set_aspect("equal")
ax_map.set_axis_off()
ax_map.set_title("Campus detail", loc="left", fontsize=13, color=COLORS["ink"], pad=8)
ax_map.text(
    0.01, 0.02,
    "Circle area = mean hidden distance",
    transform=ax_map.transAxes,
    fontsize=9,
    color=COLORS["muted"],
)

# Compact rank panel; the highest-burden destination receives a subtle emphasis.
ranked = dest_access.sort_values("mean_friction_m", ascending=True).copy()
bar_colors = [COLORS["campus"]] * len(ranked)
bar_colors[-1] = COLORS["danger"]
ax_rank.barh(ranked["short_name"], ranked["mean_friction_m"], color=bar_colors, height=0.58)
for i, (_, row) in enumerate(ranked.iterrows()):
    ax_rank.text(
        row.mean_friction_m + max(v.max() * 0.012, 5),
        i,
        f"{row.mean_friction_m:,.0f} m",
        va="center",
        fontsize=9,
        color=COLORS["ink"],
    )
ax_rank.set_title("Mean hidden distance", loc="left", fontsize=13, color=COLORS["ink"], pad=8)
ax_rank.set_xlabel("Network distance − Euclidean distance (m)")
ax_rank.grid(axis="x", alpha=0.16)
ax_rank.spines[["top", "right", "left"]].set_visible(False)
ax_rank.tick_params(axis="y", length=0)
ax_rank.set_xlim(0, v.max() * 1.20)

# Number key connects the map and ranking without overlapping long labels.
key_text = "   ".join(
    f"{int(row.map_id)}  {row.short_name}" for _, row in dest_access.sort_values("map_id").iterrows()
)
fig.suptitle(
    "Map 13. Delivery friction varies across campus destinations",
    x=0.06, y=0.98, ha="left", fontsize=18, fontweight="bold", color=COLORS["ink"]
)
fig.text(
    0.06, 0.925,
    "A campus-scale view reveals small but consequential differences hidden by the regional map.",
    ha="left", fontsize=10.5, color=COLORS["muted"]
)
fig.text(0.06, 0.035, key_text, ha="left", fontsize=9.5, color=COLORS["ink"])
fig.subplots_adjust(top=0.86, bottom=0.12, left=0.06, right=0.97)
plt.show()

## Chart 7 — Variation within each destination

The map reports destination means. This chart adds a different layer of evidence by showing the **distribution** of hidden distance across all sampled restaurant routes, preventing the average from concealing route-to-route variation.

In [ ]:
destination_order = (
    pairs.groupby("destination")["friction_m"]
    .median()
    .sort_values()
    .index
    .tolist()
)
friction_groups = [
    pairs.loc[pairs["destination"] == destination, "friction_m"].dropna().to_numpy()
    for destination in destination_order
]
labels = [short_names.get(destination, destination) for destination in destination_order]

fig, ax = plt.subplots(figsize=(10, 5.8))
box = ax.boxplot(
    friction_groups,
    vert=False,
    labels=labels,
    patch_artist=True,
    showfliers=False,
    widths=0.58,
    medianprops={"color": COLORS["ink"], "linewidth": 1.6},
    whiskerprops={"color": COLORS["muted"]},
    capprops={"color": COLORS["muted"]},
)
for patch in box["boxes"]:
    patch.set_facecolor(COLORS["campus"])
    patch.set_alpha(0.78)
    patch.set_edgecolor("white")

finish_chart(
    ax,
    "Chart 7. Similar averages can conceal different route experiences",
    "Boxes show the interquartile range; whiskers show non-outlier variation across sampled restaurant routes.",
    xlabel="Hidden distance: network distance − Euclidean distance (m)",
    ylabel=None,
)
ax.grid(axis="x", alpha=0.16)
ax.grid(axis="y", visible=False)
plt.show()

## Map 14 — High-friction origins and routes

In [ ]:
restaurant_burden=pairs.groupby("camis",as_index=False).agg(mean_friction_m=("friction_m","mean"),mean_detour=("detour_ratio","mean"))
origins=route_restaurants.merge(restaurant_burden,on="camis",how="left")
q=pairs.friction_m.quantile(.75); high=pairs[pairs.friction_m>=q]
fig,ax=plt.subplots(figsize=(11,11)); edges_proj.plot(ax=ax,color=COLORS["street"],linewidth=.16,alpha=.25)
high.plot(ax=ax,color=COLORS["danger"],linewidth=1.2,alpha=.45)
origins.plot(ax=ax,column="mean_friction_m",cmap="YlOrRd",markersize=42,edgecolor="white",linewidth=.35,legend=True,legend_kwds={"label":"Mean origin friction (m)"})
destinations_metric.plot(ax=ax,color=COLORS["campus"],markersize=75,edgecolor="white")
finish_map(ax,"Map 14. High-friction restaurants form a geography of disadvantage","Only the upper quartile of route penalties is drawn over origin-level mean friction",legend=False)
plt.show()

## 10. Test whether distance choice changes apparent proximity

In [ ]:
pairs["euclidean_rank"]=pairs.groupby("destination")["euclidean_m"].rank(method="min")
pairs["network_rank"]=pairs.groupby("destination")["network_m"].rank(method="min")
pairs["rank_shift"]=pairs.network_rank-pairs.euclidean_rank
ranking_summary=pairs.groupby("restaurant",as_index=False).agg(mean_abs_rank_shift=("rank_shift",lambda s:s.abs().mean()))
ranking_summary=ranking_summary.sort_values("mean_abs_rank_shift",ascending=False).head(15).sort_values("mean_abs_rank_shift")

## Chart 8 — Restaurants whose apparent rank changes most

In [ ]:
fig,ax=plt.subplots(figsize=(10,7)); ax.barh(ranking_summary.restaurant.str.slice(0,32),ranking_summary.mean_abs_rank_shift,color=COLORS["route"])
finish_chart(ax,"Chart 8. Straight-line ranking can misrepresent practical accessibility","Large shifts identify restaurants whose position changes after street-network constraints are introduced",xlabel="Mean absolute rank shift across destinations")
plt.show()

## Final synthesis

In [ ]:
# Compact metrics used in the written interpretation.
summary_metrics = pd.Series({
    "Restaurants in 5 km dataset": len(restaurants_gdf),
    "Restaurants routed": route_restaurants.camis.nunique(),
    "Successful route pairs": len(pairs),
    "Median Euclidean distance (m)": pairs.euclidean_m.median(),
    "Median network distance (m)": pairs.network_m.median(),
    "Median hidden distance (m)": pairs.friction_m.median(),
    "Median detour ratio": pairs.detour_ratio.median(),
    "Median modeled travel time (min)": pairs.modeled_minutes.median(),
})
display(summary_metrics.to_frame("value").round(2))

## Results interpretation: what the two distances mean

The comparison should not be interpreted as a competition between a “correct” and an “incorrect” distance. Each measure describes a different spatial experience.

**Euclidean distance** describes geometric closeness. It is useful as a baseline because it ignores the street system and asks how far apart two objects are in continuous space. This resembles the immediate visual impression produced by a map interface: two points may look close because their straight-line separation is small.

**Network distance** describes operational closeness. A delivery worker cannot travel through buildings, blocks, restricted campus edges, parks, or disconnected street segments. The rider must follow connected, traversable streets and enter the campus through available approaches. Network distance therefore better approximates the physical effort associated with an actual delivery route.

The gap between the measures—the hidden distance—is experiential. It may represent extra riding time, more intersections, additional turns, greater exposure to traffic, and a higher probability of delay. Two restaurants with similar Euclidean distance can consequently feel very different to reach. The maps and charts below locate where that difference is largest and show whether it is concentrated around particular origins, destinations, or network corridors.


## Critical reflection

This analysis shows that distance is not simply a neutral geometric quantity. Euclidean distance describes separation, but network distance describes the movement that an urban system permits. For food delivery around Columbia University, the difference becomes tangible because riders do not move through abstract space: they navigate a connected street system shaped by long blocks, one-way patterns, major avenues, parks, grade changes, campus boundaries, and a limited number of practical approaches to buildings.

The experiential difference is clearest when two restaurants appear equally close on a map but produce different routed journeys. A straight line encourages the expectation of quick access, while the network route may require movement away from the destination before turning back toward it. The additional distance may be experienced as time, physical effort, traffic exposure, uncertainty, and delivery friction. From the customer’s perspective, this infrastructure is mostly invisible; from the rider’s perspective, it structures the entire trip.

The results also complicate the idea that a campus destination has one fixed level of accessibility. Accessibility depends on where the trip begins and how the origin is connected to the surrounding network. A destination may perform well on average while still producing severe detours for origins located across a barrier or poorly connected corridor. For this reason, the distribution of route-level values is as important as a single mean.

Several limitations matter. OpenStreetMap represents mapped connectivity rather than every informal movement used by riders. The shortest-distance route may not be the route a rider actually chooses, because it does not fully model slope, traffic stress, signal delay, construction, safety, bicycle restrictions, building entrances, platform dispatch logic, or waiting time. Restaurant inspection records also indicate licensed establishments rather than real-time platform availability. The route sample is designed to represent the 5 km restaurant field without calculating every possible pair, but sampling still introduces uncertainty.

Future work could compare distance-minimizing routes with time-minimizing or low-stress routes, include elevation and intersection delay, validate paths with observed rider traces, and model campus entry points explicitly. Even with these limits, the comparison demonstrates the assignment’s central point: **Euclidean and network distance produce different understandings of proximity because one measures geometry while the other measures possible movement.**


## Conclusion

This notebook defined a weighted bicycle street network, identified restaurant-origin and campus-destination nodes, and calculated Euclidean and shortest-path network distances between them. The results demonstrate that straight-line proximity frequently understates the distance that must actually be traveled.

The most important finding is conceptual as well as numerical: **proximity depends on the system through which movement occurs**. Euclidean distance makes restaurants and destinations appear close in continuous space; network distance reveals how streets, barriers, connectivity, and access points transform that apparent closeness into an experienced journey. The resulting hidden distance is therefore not a calculation error. It is evidence of urban structure.


## Assignment checklist

- **Network clearly defined:** nodes, edges, topology, and edge weights are specified.
- **Network creation shown step by step:** data retrieval, projection, graph construction, and quality checks are documented.
- **Measured nodes identified:** restaurant origins and campus destinations are explicitly mapped and snapped to graph nodes.
- **Distances calculated:** Euclidean distance and weighted shortest-path network distance are computed for each pair.
- **Experiential difference discussed:** results are interpreted in relation to delivery movement, effort, access, and urban barriers.
- **Series of maps and charts provided:** the visual sequence moves from study area and data to network construction, routes, comparison, and reflection.


## Reproducibility and exported files

- **Restaurant source:** NYC DOHMH Restaurant Inspection Results (`43nn-pn8j`)
- **Network source:** OpenStreetMap via OSMnx
- **Study radius:** exact projected distance of 5,000 m from Columbia University
- **Routing mode:** bicycle-accessible network, shortest path weighted by edge length
- **Sampling:** proportional stratification across five 1 km distance bands
- **Exports:** `delivery_pairs_5km.csv`, `delivery_routes_5km.geojson`, and `destination_accessibility_5km.csv`

All figures are generated from the notebook workflow. No figure is included only to reach a numerical quota; each one performs a distinct step in the argument.